# Diffusion on Colab T4

SD3.5-medium, 4-bit quantized, txt2img + img2img + inpaint.

**Before running:** Runtime -> Change runtime type -> T4 GPU.

Settings live in `config.yaml`, not in this notebook. Edit that file to change models or generation params.

## 0. Get the repo files into this session

If you opened this notebook directly (not via `git clone`), upload `config.yaml` into the Colab file browser (left sidebar) now, or clone your repo below.

In [ ]:
# Uncomment and edit once you've pushed this repo to GitHub:
# !git clone https://github.com/<your-username>/diffusion-colab-t4.git
# %cd diffusion-colab-t4

import os
assert os.path.exists("config.yaml"), "config.yaml not found — upload it or clone the repo first."

## 1. Confirm GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch
assert torch.cuda.is_available(), "No GPU detected — check Runtime > Change runtime type > T4 GPU"
print("GPU OK:", torch.cuda.get_device_name(0))

## 2. Install dependencies

In [ ]:
!pip install -q -U -r requirements.txt

## 3. Load config

In [ ]:
import yaml

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg

## 4. Hugging Face auth

SD3.5 is a gated model — you must accept the license on the model page once
(https://huggingface.co/stabilityai/stable-diffusion-3.5-medium) before this works.

Store your token in Colab Secrets (key icon, left sidebar) as `HF_TOKEN`. Don't paste it into a cell.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

## 5. Load the model (NF4 quantized) once

We load a single text2img pipeline, then derive img2img and inpaint pipelines
from it with `from_pipe()` — this reuses the same weights in VRAM instead of
loading the model three times, which is essential at 16GB.

In [ ]:
import torch
from transformers import T5EncoderModel
from diffusers import (
    BitsAndBytesConfig,
    SD3Transformer2DModel,
    AutoPipelineForText2Image,
    AutoPipelineForImage2Image,
    AutoPipelineForInpainting,
)

model_id = cfg["model"]["id"]
dtype = getattr(torch, cfg["model"]["dtype"])

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype,
)

# device_map="cuda:0" quantizes each shard directly on the GPU as it streams
# in, instead of first materializing full-precision weights in CPU RAM.
transformer = SD3Transformer2DModel.from_pretrained(
    model_id,
    subfolder="transformer",
    quantization_config=quant_config,
    torch_dtype=dtype,
    device_map="cuda:0",
    low_cpu_mem_usage=True,
)

# This is the actual fix for the RAM crash: SD3.5 loads THREE text encoders,
# and the third one (T5-XXL) is ~9GB unquantized in bf16 -- roughly as large
# as the transformer itself. Quantizing it the same way we quantized the
# transformer is what brings total load-time memory back under budget.
text_encoder_3 = T5EncoderModel.from_pretrained(
    model_id,
    subfolder="text_encoder_3",
    quantization_config=quant_config,
    torch_dtype=dtype,
    device_map="cuda:0",
    low_cpu_mem_usage=True,
)

pipe_t2i = AutoPipelineForText2Image.from_pretrained(
    model_id,
    transformer=transformer,
    text_encoder_3=text_encoder_3,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
)

if cfg["memory"]["enable_cpu_offload"]:
    pipe_t2i.enable_model_cpu_offload()
if cfg["memory"]["enable_vae_slicing"]:
    pipe_t2i.enable_vae_slicing()
if cfg["memory"]["enable_vae_tiling"]:
    pipe_t2i.enable_vae_tiling()

# Derive img2img and inpaint pipelines from the same loaded weights
pipe_i2i = AutoPipelineForImage2Image.from_pipe(pipe_t2i)
pipe_inpaint = AutoPipelineForInpainting.from_pipe(pipe_t2i)

print("Pipelines ready.")

## 6. Output helper

In [ ]:
import os
from datetime import datetime
from IPython.display import display

out_dir = cfg["output"]["dir"]
os.makedirs(out_dir, exist_ok=True)

def save_and_show(image, prefix="img"):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = os.path.join(out_dir, f"{prefix}_{ts}.{cfg['output']['save_format']}")
    image.save(path)
    display(image)
    print("Saved to", path)
    return path

## 7. Text-to-image

In [ ]:
prompt = "a weathered lighthouse on a cliff at dusk, cinematic lighting, detailed"
params = cfg["generation"]["txt2img"]

image = pipe_t2i(
    prompt=prompt,
    num_inference_steps=params["num_inference_steps"],
    guidance_scale=params["guidance_scale"],
    height=params["height"],
    width=params["width"],
).images[0]

save_and_show(image, prefix="t2i")

## 8. Image-to-image

Reuses `image` from the cell above, or load your own with `Image.open("path.png")`.

In [ ]:
from PIL import Image

init_image = image  # or: Image.open("your_input.png").convert("RGB")
i2i_prompt = "same lighthouse, but during a storm, dramatic waves"
params = cfg["generation"]["img2img"]

result = pipe_i2i(
    prompt=i2i_prompt,
    image=init_image,
    strength=params["strength"],
    num_inference_steps=params["num_inference_steps"],
    guidance_scale=params["guidance_scale"],
).images[0]

save_and_show(result, prefix="i2i")

## 9. Inpainting

You need a source image and a mask (white = repaint, black = keep).
Draw a mask quickly in any image editor, or generate one programmatically.

In [ ]:
# Example: replace with your own image + mask paths
# source = Image.open("source.png").convert("RGB")
# mask = Image.open("mask.png").convert("RGB")

source = image
mask = Image.new("RGB", source.size, (0, 0, 0))  # placeholder: all-black mask (no-op)

inpaint_prompt = "a small wooden boat on the water"
params = cfg["generation"]["inpaint"]

result = pipe_inpaint(
    prompt=inpaint_prompt,
    image=source,
    mask_image=mask,
    strength=params["strength"],
    num_inference_steps=params["num_inference_steps"],
    guidance_scale=params["guidance_scale"],
).images[0]

save_and_show(result, prefix="inpaint")